In [2]:
!pip install numpy pandas

In [3]:
import numpy as np
import pandas as pd
from math import log

# DNA Region Prediction using Hidden Markov Models

This notebook demonstrates how Hidden Markov Models can identify hidden biological regions from DNA sequences.

The Viterbi decoding algorithm is used to determine the most probable sequence of hidden states.

In [4]:
hidden_states = {
    "Start":0,
    "Exon":1,
    "Splice":2,
    "Intron":3,
    "Stop":4
}

reverse_state = {
    v:k
    for k,v in hidden_states.items()
}

hidden_states

{'Start': 0, 'Exon': 1, 'Splice': 2, 'Intron': 3, 'Stop': 4}

In [5]:
transition_matrix=np.array([

[0,1,0,0,0],

[0,.9,.1,0,0],

[0,0,0,1,0],

[0,0,0,.9,.1],

[0,0,0,0,0]

])

pd.DataFrame(
transition_matrix,
index=hidden_states.keys(),
columns=hidden_states.keys()
)

,Start,Exon,Splice,Intron,Stop
Start,0.0,1.0,0.0,0.0,0.0
Exon,0.0,0.9,0.1,0.0,0.0
Splice,0.0,0.0,0.0,1.0,0.0
Intron,0.0,0.0,0.0,0.9,0.1
Stop,0.0,0.0,0.0,0.0,0.0


In [6]:
nucleotide_index={

"A":0,
"C":1,
"G":2,
"T":3

}

emission_matrix=np.array([

[0,0,0,0],

[0.25,0.25,0.25,0.25],

[0.05,0,0.95,0],

[0.4,0.1,0.1,0.4],

[0,0,0,0]

])

pd.DataFrame(
emission_matrix,
index=hidden_states.keys(),
columns=["A","C","G","T"]
)

,A,C,G,T
Start,0.00,0.00,0.00,0.00
Exon,0.25,0.25,0.25,0.25
Splice,0.05,0.00,0.95,0.00
Intron,0.40,0.10,0.10,0.40
Stop,0.00,0.00,0.00,0.00


In [7]:
dna_sequence="ATGCGTTAGCGATCGATTCGA"
print(dna_sequence)

ATGCGTTAGCGATCGATTCGA


In [8]:
state_count=len(hidden_states)

sequence_size=len(dna_sequence)

score_matrix=np.full(
(state_count,sequence_size),
-np.inf
)

path_matrix=np.zeros(
(state_count,sequence_size),
dtype=int
)

In [9]:
first_base=dna_sequence[0]

score_matrix[1][0]=(
log(1)
+
log(
emission_matrix[1][
nucleotide_index[first_base]
]
)
)

In [10]:
def evaluate_state(current_state,column):

    best_score=-np.inf
    previous_best=0

    current_base=dna_sequence[column]

    for prev_state in range(state_count):

        t_prob=transition_matrix[
            prev_state
        ][current_state]

        e_prob=emission_matrix[
            current_state
        ][
            nucleotide_index[current_base]
        ]

        if t_prob==0 or e_prob==0:
            continue

        candidate=(
            score_matrix[prev_state][column-1]
            +log(t_prob)
            +log(e_prob)
        )

        if candidate>best_score:

            best_score=candidate
            previous_best=prev_state

    return best_score,previous_best

In [11]:
for column in range(
1,
sequence_size
):

    for state in range(
state_count
):

        value,prev=evaluate_state(
        state,
        column
        )

        score_matrix[
        state
        ][column]=value

        path_matrix[
        state
        ][column]=prev

In [12]:
def decode_path():

    column=sequence_size-1

    state=np.argmax(
    score_matrix[:,column]
    )

    path=[state]

    while column>0:

        state=path_matrix[
        state
        ][column]

        path.append(state)

        column-=1

    path.reverse()

    return [
    reverse_state[x]
    for x in path
    ]

In [13]:
prediction=decode_path()

print("Observed DNA:")
print(dna_sequence)

print()

print("Predicted regions:")
print(prediction)

Observed DNA:
ATGCGTTAGCGATCGATTCGA

Predicted regions:
['Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon', 'Exon']
